# TinyCast: GIFT-Eval reproduction

This notebook is the GIFT-Eval leaderboard `code_link` for **TinyCast**, an
attention-free 146,505-parameter dilated-convolution time-series foundation
model. It reproduces the published aggregates from the pinned per-configuration
results, loads the released weights, forecasts, and runs the 97 GIFT-Eval
configurations zero-shot into a leaderboard-format `all_results.csv`.

- Model: https://huggingface.co/raws-labs/tinycast
- Code: https://github.com/raws-labs/tinycast

It has two jobs. Copy it as a starting point and delete the assertion block at
the end of each cell; run it unchanged and those assertions say whether the
installed package still reproduces the paper.

**Scale.** `TINYCAST_NB_SCALE` picks how much of the benchmark section 6 covers:
`full` (the default) is all 97 configurations, `smoke` is one small one. Both
settings run the same lines; only the configuration count differs.

**Requirements.**
`pip install git+https://github.com/raws-labs/tinycast.git`
covers everything except section 6,
which also wants the GIFT-Eval data loader and the benchmark data at
`$GIFT_EVAL`. The loader is not on PyPI and its distribution name is not its
import name, so both have to be spelled out:

```
pip install "salesforce-gift-eval @ git+https://github.com/SalesforceAIResearch/gift-eval.git"
```

installs it; `import gift_eval` imports it. Sections 1, 2 and 5 need neither a
GPU nor the network.

**Two kinds of missing input, two different outcomes.** Sections 3, 4 and 6 need
the released weights, and an unresolvable weight file is a failure, not a skip:
`model.safetensors` is 0.6 MB and ships at the root of the supplementary
package, so `TINYCAST_WEIGHTS` pointed at that root resolves it with no network,
and the hub copy resolves it otherwise. A reader who can reach neither has a
broken environment, and a run that verified none of the weight-dependent claims
must not report success. Section 6 additionally needs the benchmark data, which
is a large download and is not assumed present; without it that section skips.
Both paths announce themselves; neither passes quietly.


In [ ]:
import os

# Scale. "full" (the default) evaluates all 97 GIFT-Eval configurations in
# section 6; "smoke" evaluates one. Nothing else changes.
SCALE = os.environ.get("TINYCAST_NB_SCALE", "full").strip().lower()
N_CONFIGS = {"full": 97, "smoke": 1}.get(SCALE)

# The GIFT-Eval data directory: the parent of the per-dataset folders.
os.environ.setdefault("GIFT_EVAL", "/path/to/gift-eval")
# Thread caps, set before torch is imported below. Without them
# per-configuration evaluation is roughly 40 times slower on a many-core host.
for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
            "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(var, "4")


def loud(headline, *lines):
    """Announce a skip or an opt-out so it cannot be read as a pass."""
    bar = "!" * 74
    print("\n".join([bar, headline, *lines, bar]))


# Two environment variables change what the predictor emits, and both are off in
# the published run: TINYCAST_INT8 applies post-training fake quantization and
# TINYCAST_TILT_K tilts the median away from the quantile grid. While either is
# in effect, the golden array in section 4 is not the published model's forecast.
# What counts as in effect is the predictor's own rule (TINYCAST_INT8 acts at
# "w8" and "w8a8" and refuses a value it cannot interpret, TINYCAST_TILT_K is a
# float that does nothing at 0, and TINYCAST_TILT_MODE is read only when the
# tilt is on), so this asks the predictor rather than restating the rule and
# drifting from it. Either resolver raises on a value the predictor would refuse.
from tinycast.predictor import _resolve_int8, _resolve_tilt

int8_mode = _resolve_int8()
tilt_k, tilt_mode = _resolve_tilt()
SWITCHES = []
if int8_mode:
    SWITCHES.append(f"TINYCAST_INT8={int8_mode}")
if tilt_k:
    SWITCHES.append(f"TINYCAST_TILT_K={tilt_k:g} ({tilt_mode})")

print(f"scale={SCALE!r}, configurations={N_CONFIGS}")
print(f"predictor switches in effect: {SWITCHES or 'none'}")

assert N_CONFIGS is not None, f"TINYCAST_NB_SCALE must be 'full' or 'smoke', got {SCALE!r}"
assert not SWITCHES, f"unset {SWITCHES} to reproduce the published numbers"


## 1. Reproduce the published aggregates offline

The three headline numbers are geometric means, over the 97 leaderboard
configurations, of the ratio between the model's metric and the seasonal-naive
reference's. The per-configuration results behind them ship with the package, so
re-deriving the aggregates takes about a millisecond and needs no GPU, no
benchmark data and no network. `summarize_by_freq_bin` is the same function the
benchmark driver calls at the end of a run, so this also checks that the
reference file and the summarizer still agree.

`nWQL` is the leaderboard's name for the quantile-loss aggregate and `nCRPS` is
the paper's; the summary carries the one number under both keys.

In [ ]:
from pathlib import Path

import tinycast
from tinycast import summarize_by_freq_bin

PINNED = Path(tinycast.__file__).parent / "reference" / "gift_eval_tinycast.csv"

# Reading a file must not write one: the .bins.json sidecar is written only on
# write_sidecar=True, which the benchmark driver passes and this replay does not.
# So the pinned results are read where they are installed and nothing lands
# beside them.
published = summarize_by_freq_bin(PINNED)

overall = published["overall"]
print(f"nGMASE {overall['ngmase']:.10f}")
print(f"nWQL   {overall['nwql']:.10f}")
print(f"nMSIS  {overall['nmsis']:.10f}")

assert overall["n"] == 97
assert published["dropped"]["n_dropped"] == 0
assert overall["n_ngmase"] == 97
assert overall["n_ncrps"] == 97
assert overall["n_nmsis"] == 97
assert round(overall["ngmase"], 4) == 0.7738
assert round(overall["ncrps"], 4) == 0.5454
assert round(overall["nmsis"], 4) == 0.5541
assert overall["nwql"] == overall["ncrps"]
assert "sidecar" not in published
assert not PINNED.with_suffix(".bins.json").exists()

## 2. The parameter count, and the trap in a `state_dict`

Both FFN stacks are weight-tied. `state_dict()` materializes the shared storage
once per alias, so summing its tensors reports 321,225 values for a model that
holds 146,505. Take the count from a model instantiated from its config, never
from a sum of checkpoint tensors. The defaults of `TinyCastConfig` are the
released model's, so no file is needed here.

In [ ]:
from tinycast import TinyCastConfig, TinyCastForPrediction

config = TinyCastConfig()
model = TinyCastForPrediction(config)

n_parameters = sum(p.numel() for p in model.parameters())
state_dict_sum = sum(t.numel() for t in model.state_dict().values())
print(f"instantiated from config: {n_parameters:,} parameters")
print(f"summed over state_dict(): {state_dict_sum:,} values "
      f"({state_dict_sum - n_parameters:,} of them tied aliases)")

assert n_parameters == 146_505
assert state_dict_sum == 321_225

## 3. The released weights

The release is a `model.safetensors` (0.6 MB) plus a `config.json`.
`load_model` rebuilds the architecture from the config and restores the FFN
weight sharing on load.

Set `TINYCAST_WEIGHTS` to a local `model.safetensors`, or to the directory
holding it and its `config.json` sibling, to run this notebook with no network
at all. That variable is read here, not by the package.

**Unresolvable weights fail here.** Everyone who runs this notebook can reach
the file: it ships at the root of the supplementary package, and the hub copy is
one download away. So an unresolved path is a broken environment, and failing is
the only honest outcome; skipping would let a run that checked nothing about the
weights finish green, which is worse than a red one.

`TINYCAST_SKIP_WEIGHTS=1` opts out for the case where only the
weight-independent cells (1, 2 and 5) are wanted. It downgrades the failure to a
skip and says, at every step it skips, that the weight-dependent claims went
unverified.


In [ ]:
from tinycast import load_model

HF_REPO = "raws-labs/tinycast"
# The deliberate opt-out. It does not make an unresolvable weight file
# acceptable; it records that nothing about the weights was checked.
SKIP_WEIGHTS = os.environ.get("TINYCAST_SKIP_WEIGHTS", "").strip().lower() in (
    "1", "true", "yes", "on")

REMEDY = (
    "  Set TINYCAST_WEIGHTS to a local model.safetensors, or to the directory\n"
    "  holding it and its config.json sibling. The supplementary package ships\n"
    "  model.safetensors at its root, so pointing TINYCAST_WEIGHTS at that root\n"
    "  resolves it with no network at all.\n"
    f"  Otherwise the file comes from https://huggingface.co/{HF_REPO}.\n"
    "  TINYCAST_SKIP_WEIGHTS=1 runs the weight-independent cells only and leaves\n"
    "  every weight-dependent claim unverified."
)


def resolve_weights():
    """A local copy if TINYCAST_WEIGHTS names one, else the hub copy.

    Returns (path, None) on success and (None, reason) when neither is reachable.
    """
    local = os.environ.get("TINYCAST_WEIGHTS", "").strip()
    if local:
        path = Path(local)
        path = path / "model.safetensors" if path.is_dir() else path
        if not path.is_file():
            raise FileNotFoundError(
                f"TINYCAST_WEIGHTS={local!r} is not a model.safetensors\n{REMEDY}"
            )
        return str(path), None
    try:
        from huggingface_hub import hf_hub_download

        hf_hub_download(HF_REPO, "config.json")
        return hf_hub_download(HF_REPO, "model.safetensors"), None
    except Exception as exc:
        return None, (f"TINYCAST_WEIGHTS is unset and the hub copy is "
                      f"unreachable: {type(exc).__name__}: {exc}")


WEIGHTS, unresolved = resolve_weights()
loaded_parameters, loaded_config = None, None
if WEIGHTS is None and not SKIP_WEIGHTS:
    raise RuntimeError(f"the released weights could not be resolved.\n"
                       f"  {unresolved}\n{REMEDY}")
if WEIGHTS is None:
    loud("TINYCAST_SKIP_WEIGHTS is set: the released weights were NOT loaded.",
         f"  {unresolved}",
         "Sections 4 and 6 will not run. The released file's parameter count,",
         "the golden forecast and the benchmark aggregates are NOT verified,",
         "so a green run under this variable says nothing about the weights.")
else:
    loaded, loaded_config = load_model(WEIGHTS)
    loaded_parameters = sum(p.numel() for p in loaded.parameters())
    print(f"loaded {WEIGHTS}")
    print(f"{loaded_parameters:,} parameters, context {loaded_config.seq_len}, "
          f"horizon {loaded_config.output_token_len}, "
          f"{loaded_config.num_quantiles} quantiles")

assert WEIGHTS is not None or SKIP_WEIGHTS
assert WEIGHTS is None or loaded_parameters == 146_505
assert WEIGHTS is None or loaded_config.to_dict() == config.to_dict()


## 4. A golden forecast

`TinyCastPredictor` is the gluonts predictor the leaderboard entry was produced
with: autoregressive rollout in 48-step chunks, flip-invariance symmetrization,
NaN imputation and period-alignment downsampling. It takes the safetensors path
and picks up the `config.json` sibling itself.

The context below is fixed, so the median forecast is too. The comparison is not
bitwise: CPU convolution kernels differ across builds and thread counts, and the
observed spread is around 1e-6, so `atol=1e-4` catches a changed model while
tolerating a changed BLAS.

**Assert the resolved device, not the requested one.** The golden values are CPU
fp32, and a CUDA run under bf16 autocast sits about 1e-3 away, which this
tolerance would not absorb. A named device that is absent raises here rather
than being quietly substituted, and `device=None` is the one request that lets
the predictor choose, so `predictor.device` is the record of what ran.

In [ ]:
import numpy as np
import pandas as pd

from tinycast import TinyCastPredictor

DEVICE = "cpu"                    # the golden values below are CPU fp32
GOLDEN_MEDIAN = [7.447377, 7.218568, 7.172349, 7.309259,
                 7.557366, 8.108236, 8.755529, 9.532342]


def demo_context(length=1024):
    """A daily cycle, a weekly cycle and fixed noise: a fully determined input."""
    rng = np.random.default_rng(0)
    t = np.arange(length)
    series = (10.0
              + 3.0 * np.sin(2 * np.pi * t / 24)
              + 0.5 * np.sin(2 * np.pi * t / 168)
              + 0.2 * rng.standard_normal(length))
    return series.astype("float32")


series = demo_context()
start = pd.Period("2020-01-01 00", freq="h")
entry = {"target": series, "start": start, "item_id": "demo"}

predictor, median = None, None
if WEIGHTS is None:
    loud("skipped under TINYCAST_SKIP_WEIGHTS: no released weights.",
         "The golden forecast was NOT compared against the published model.")
else:
    predictor = TinyCastPredictor(
        prediction_length=48, checkpoint_path=WEIGHTS,
        freq="H", domain="Energy", device=DEVICE,
        force_flip_invariance=True,
    )
    median = predictor.predict([entry])[0].quantile(0.5)
    print(f"requested device {DEVICE!r}, resolved {str(predictor.device)!r}")
    print("median forecast, first 8 steps:", np.round(median[:8], 6).tolist())

assert predictor is None or str(predictor.device) == DEVICE
assert median is None or median.shape == (48,)
assert median is None or np.allclose(median[:8], GOLDEN_MEDIAN, rtol=0, atol=1e-4)


## 5. Flip-invariance symmetrization is exact

With `force_flip_invariance=True` the predictor averages the forecast with the
quantile-reversed forecast of the negated input. That makes the predictor odd:
the tau-quantile of the forecast for `-x` is minus the (1-tau)-quantile of the
forecast for `x`, and in IEEE arithmetic the identity is exact rather than
approximate.

It is a property of the symmetrization, not of the weights, so this cell uses a
freshly initialized model and needs no network. Exporting it first also
exercises `export_safetensors`, which refuses to write unless the instantiated
model has the expected parameter count.

In [ ]:
import tempfile

import torch

from tinycast import export_safetensors

torch.manual_seed(0)
fresh = TinyCastForPrediction(TinyCastConfig())

with tempfile.TemporaryDirectory() as scratch:
    written = export_safetensors(fresh, scratch, expect_parameters=146_505)
    symmetric = TinyCastPredictor(
        prediction_length=48, checkpoint_path=str(written["weights_path"]),
        freq="H", domain="Energy", device="cpu",
        force_flip_invariance=True,
    )
    up = symmetric.predict([{"target": series, "start": start, "item_id": "up"}])
    down = symmetric.predict([{"target": (-series).astype("float32"),
                               "start": start, "item_id": "down"}])

# forecast_array is (Q, prediction_length) with the quantile levels ascending.
forward = up[0].forecast_array
negated = down[0].forecast_array
print(f"{written['num_tensors']} tensors written, "
      f"{written['num_tied_aliases']} tied aliases recorded")
print("max |F(x) + reverse(F(-x))| =", np.abs(forward + negated[::-1]).max())

assert forward.shape == (config.num_quantiles, 48)
assert np.array_equal(forward, -negated[::-1])

## 6. The leaderboard run

`evaluate` writes one row per configuration in the leaderboard column order and
returns the summary it computed, so the aggregates can be asserted without
re-reading the file. It also writes an `all_results.bins.json` sidecar carrying
the per-frequency-bin values of all three normalized aggregates.

Any configuration a reference lookup or a missing value would have dropped from
an aggregate raises instead of being skipped: a geometric mean keeps no record of
how many terms it has, so a drifted reference would otherwise shrink the sample
and still return a publishable-looking number.

The published aggregates are the CUDA run under bf16 autocast, the leaderboard's
`bfloat16` setting. On CPU the driver runs fp32 and the numbers move, so the
tolerance below depends on which device actually ran. That comparison needs all
97 configurations: a geometric mean over a subset is a different quantity and no
tolerance relates the two, so at `smoke` scale the cell reports the subset
aggregates and checks what a subset can check, that the run completes with all
three metrics present and nothing dropped.

The benchmark data is a large download, so this section skips when it or the
`gift_eval` loader is absent. It skips loudly: sections 3 and 4 still verified
the weights, and the leaderboard run is then the one claim the notebook did not.


In [ ]:
import torch

from tinycast.eval import evaluate, iter_configs

PUBLISHED = {"ngmase": 0.7737836631366714,
             "ncrps": 0.5454197284419108,
             "nmsis": 0.5541246851433915}

CONFIG_KEYS = [key for _, _, key, _, _ in iter_configs()]
# A smoke run takes a prefix, and the first key in dataset order is one of the
# most expensive configurations there is, so three small ones are moved to the
# front. A full run takes all 97 and the order makes no difference.
CHEAP_FIRST = ("m4_hourly/H/short", "hospital/M/short", "covid_deaths/D/short")
ordered = ([k for k in CHEAP_FIRST if k in CONFIG_KEYS]
           + [k for k in CONFIG_KEYS if k not in CHEAP_FIRST])
wanted = None if N_CONFIGS == len(ordered) else ordered[:N_CONFIGS]
EVAL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# On CUDA the run should land on the published numbers to rounding. On CPU it is
# a different arithmetic path, so the check is only that nothing gross broke.
TOLERANCE = 1e-4 if EVAL_DEVICE == "cuda" else 2e-2
GIFT_EVAL_INSTALL = ('pip install "salesforce-gift-eval @ '
                     'git+https://github.com/SalesforceAIResearch/gift-eval.git"')


def benchmark_blocker():
    """Why the benchmark cannot run here, or None when it can."""
    if WEIGHTS is None:
        return "the released weights were not loaded (TINYCAST_SKIP_WEIGHTS)"
    try:
        import gift_eval  # noqa: F401
    except ImportError:
        return f"gift_eval is not installed; {GIFT_EVAL_INSTALL}"
    root = Path(os.environ.get("GIFT_EVAL", ""))
    if not root.is_dir():
        return f"GIFT_EVAL={str(root)!r} is not a directory"
    return None


blocker = benchmark_blocker()
summary, worst_gap = None, 0.0
if blocker is not None:
    loud(f"skipped: {blocker}",
         "The leaderboard run did NOT execute, so nothing here checked the",
         "published aggregates against a live evaluation. Section 1 replayed",
         "them from the pinned per-configuration results instead.")
else:
    summary = evaluate(
        ckpt_path=WEIGHTS,
        output_csv="all_results.csv",
        model_name="TinyCast",
        configs_filter=wanted,            # None means all 97
        force_flip_invariance=True,
        device=EVAL_DEVICE,
        period_align=True,
    )
    overall_run = summary["overall"]
    print(f"{EVAL_DEVICE}, {overall_run['n']} of 97 configurations: "
          f"nGMASE {overall_run['ngmase']:.6f}, nWQL {overall_run['nwql']:.6f}, "
          f"nMSIS {overall_run['nmsis']:.6f}")
    if overall_run["n"] == 97:
        worst_gap = max(abs(overall_run[k] / v - 1) for k, v in PUBLISHED.items())
        print(f"worst relative gap to the published aggregates {worst_gap:.2e} "
              f"(tolerance {TOLERANCE:.0e})")
    else:
        print("a subset aggregate is not comparable to the published "
              "97-configuration one, so no tolerance applies; checked here: "
              "every requested configuration ran, produced all three metrics, "
              "and none was dropped.")

assert len(CONFIG_KEYS) == 97
assert summary is None or summary["overall"]["n"] == N_CONFIGS
assert summary is None or summary["dropped"]["n_dropped"] == 0
assert summary is None or summary["overall"]["n_ngmase"] == N_CONFIGS
assert summary is None or summary["overall"]["n_ncrps"] == N_CONFIGS
assert summary is None or summary["overall"]["n_nmsis"] == N_CONFIGS
assert summary is None or N_CONFIGS < 97 or worst_gap < TOLERANCE


A full run leaves an `all_results.csv` of 98 lines including the header and 15
columns, ready for a GIFT-Eval leaderboard pull request alongside
`gift_eval_submission/config.json`. Move it next to that file before opening
one.

To evaluate a hand-picked subset instead of a prefix, pass
`configs_filter=["m4_hourly/H/short", "ett1/15T/long"]` directly; the keys are
the `dataset` column of the CSV, and `iter_configs()` lists all 97.